## Malthusian Dynamics

ECO 331 Economic History (Conning)

### Model

Population $N$ and income $Y$ are related by the following system of differential equations:

$$
\begin{align*}
\frac{dN}{dt} &= N \cdot (b(y) - d(y)) \\
\frac{dY}{dt} &= g - cN
\end{align*}
$$



where:
- $y = \frac{Y}{N}$ (per capita resources)
- $b(y) = b_0y$ (birth rate increases linearly with resources)
- $d(y) = d_0e^{-\alpha y}$ (death rate decreases exponentially with resources)

Substituting these in gives the full system:

$
\frac{dN}{dt} = N(b_0\frac{Y}{N} - d_0e^{-\alpha \frac{Y}{N}}) \\
$

$
\frac{dY}{dt} = g - cN
$

Parameters:
- $b_0$ : base birth rate
- $d_0$ : base death rate
- $\alpha$ : sensitivity of death rate to resources
- $g$ : resource growth rate
- $c$ : resource consumption rate per individual
- $N$ : population size
- $Y$ : total resources




It's not very easy to find a closed-form (mathematical expression) that gives the time-path solution to this system of differential equations. However, by providing initial values for the population and total resources, we can use Python methods, such as `scipy.integrate.odeint`, to find numerical approximations to these solutions. This allows us to observe how the optimal time-path responds to changes in the model parameters.

Run all the python cells below and then adjust the sliders on the graph below to see how the model responds to different parameters.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from IPython.display import display, clear_output

In [2]:
def plot_both(b0=0.7, d0=0.15, alpha=0.2, g=100.0, c=0.1):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Left plot - Birth and Death rates
    y = np.linspace(0, 1, 100)
    birth_rate = b0 * y
    death_rate = d0 * np.exp(-alpha * y)

    ax1.plot(y, birth_rate, 'b-', label=r'$b(y) = b_0 y$')
    ax1.plot(y, death_rate, 'r-', label=r'$d(y) = d_0 e^{-\alpha y}$',
             alpha=0.7) # Added alpha for visibility if lines overlap

    # Find intersection
    intersections = np.abs(birth_rate - death_rate)
    intersection_idx = np.argmin(intersections)
    y_equilibrium = y[intersection_idx]
    rate_equilibrium = birth_rate[intersection_idx]

    ax1.plot(y_equilibrium, rate_equilibrium, 'ko',
             label=f'Equilibrium at y \u2248 {y_equilibrium:.2f}')

    ax1.set_xlabel('Per capita resources (y = Y/N)')
    ax1.set_ylabel('Rate')
    ax1.set_title('Birth and Death Rates')
    ax1.set_ylim(0, 1.5)  # Set y-axis limits for left plot
    ax1.grid(True)
    ax1.legend()

    # Right plot - Population dynamics
    def dynamics(state, t, b0, d0, alpha, g, c):
        N, Y = state
        r = Y/N if N > 0 else Y
        birth_rate = b0 * r
        death_rate = d0 * np.exp(-alpha * r)
        dNdt = N * (birth_rate - death_rate)
        dYdt = g - c*N
        return [dNdt, dYdt]


    t = np.linspace(0, 100, 1000)
    N0, Y0 = 1.0, 2.0
    state0 = [N0, Y0]

    solution = odeint(dynamics, state0, t, args=(b0, d0, alpha, g, c))

# Create second y-axis
    ax2_twin = ax2.twinx()

    # Plot population on left y-axis
    pop_line = ax2.plot(t, solution[:, 0], 'b-', label='Population')[0]
    ax2.set_xlabel('Time')
    ax2.set_ylabel('Population (N)', color='b')
    ax2.tick_params(axis='y', labelcolor='b')
    ax2.set_ylim(0, 1400)

    # Plot per capita resources on right y-axis
    per_capita = solution[:, 1] / solution[:, 0]
    res_line = ax2_twin.plot(t, per_capita, 'r-', label='Per capita resources')[0]

    eq_line = ax2_twin.axhline(y=y_equilibrium, color='k', linestyle='--',
                              label=f'Equilibrium (r = {y_equilibrium:.2f})')

    ax2_twin.set_ylabel('Per capita resources (Y/N)', color='r')
    ax2_twin.tick_params(axis='y', labelcolor='r')
    ax2_twin.set_ylim(0, 1)

    # Add combined legend
    lines = [pop_line, res_line]
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='upper right')

    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [3]:
interact(plot_both,
         b0=(0.05, 1.0, 0.01), # (min, max, step)
         d0=(0.05, 1.0, 0.01),
         alpha=(0.1, 1.0, 0.1),
         g=(0.0, 120.0, 1.0),
         c=(0.0, 0.2, 0.001));

interactive(children=(FloatSlider(value=0.7, description='b0', max=1.0, min=0.05, step=0.01), FloatSlider(valu…